#### importing libraries / configurations

In [1]:
# uncomment the line below to install required packages
# !pip install pyalex pandas numpy requests json ast itertools matplotlib seaborn

In [1]:
import requests
import ast
from itertools import chain
import pandas as pd
import pyalex
pyalex.config.email = "*useemail*"



In [2]:
from pyalex import config

config.max_retries = 3
config.retry_backoff_factor = 0.5
config.retry_http_codes = [429, 500, 503]


In [3]:
from pyalex import Works, Authors, Sources, Institutions, Topics, Publishers, Funders

#### search queries

In [4]:

EU_CODES = [
    "AT", "BE", "BG", "HR", "CY", "CZ", "DK", "EE", "FI", "FR", "DE",
    "GR", "HU", "IE", "IT", "LV", "LT", "LU", "MT", "NL", "PL", "PT",
    "RO", "SK", "SI", "ES", "SE",
]
eu_codes_str = "|".join(code.lower() for code in EU_CODES)

In [8]:
# base query for ALL layer

search_string = (
    '"AI in education" OR "artificial intelligence in education" OR "intelligent tutoring system" OR "intelligent tutoring systems" OR "intelligent tutoring" OR "educational data mining" OR "learning analytics" OR "student modeling" OR "student modelling" OR "adaptive learning system" OR "adaptive learning systems" OR "AI-based tutoring" OR "AI-powered tutoring" OR "generative AI in education" OR "LLM in education" OR "large language model in education" OR "ChatGPT in education" '
)





base_query = (
    Works()
    .search(search_string)   # TODO: later refine/search_filter/keyword list
    .filter(publication_year="2020-2025")
    .filter(authorships={"institutions": {"country_code": eu_codes_str}})
)

In [9]:
fields_works = [
    "id",
    "doi",
    "title",
    "abstract_inverted_index",   # PyAlex converts to `abstract` property
    "publication_year",
    "publication_date",
    "open_access",
    "type",
    "language",
    "cited_by_count",
    "open_access",
    "primary_location",
    "best_oa_location",
    "primary_topic",
    "topics",
    "locations",
    "concepts",
    "authorships",
    "referenced_works",
    "countries_distinct_count",
    "keywords",
    "counts_by_year",
]
################__not using the following fields for now__#####################
fields_authors = [
    "id",
    "display_name",
    "orcid",
    "affiliations",
    "topics",
    "last_known_institution",
    "works_count",
    "counts_by_year",
    "cited_by_count",
    "summary_stats",
    "topic_share",
    "x_concepts",
]

fields_institutions = [
    "id",
    "display_name",
    "country_code",
    "type",
    "works_count",
    "cited_by_count",
    "roles",
    "topics",
    "associated_institutions",
]

fields_sources = [
    "id",
    "issn_l",
    "issn",
    "display_name",
    "is_oa",
    "is_in_doaj",
    "works_count",
    "cited_by_count",
    "is_indexed_in_scopus",
    "type",
    "topics",
    
]

fields_topics = [

    "id",
    "display_name",
    "keywords",
    "works_count",
    "cited_by_count",
]

query = base_query.select(fields_works)

In [ ]:
# Function to fetch all works and convert to DataFrame
#max per_page=200, n_max can be set to None for all  records, default n_max=1000
def fetch_all_works_to_df(query, per_page=200, n_max=None):
    pages = query.paginate(per_page=per_page, n_max=n_max)
    all_records = []
    for page in pages:
        all_records.extend(page)
    return pd.DataFrame(all_records)

In [11]:

df_all = fetch_all_works_to_df(query, per_page=200, n_max=None)
print(df_all.shape)
df_all.to_csv("open_alex_data.csv", index=False)

(9689, 21)


In [12]:
df = pd.read_csv("open_alex_data.csv")
print(df.shape)
df.head()


(9689, 21)


,id,doi,title,abstract_inverted_index,publication_year,publication_date,open_access,type,language,cited_by_count,...,best_oa_location,primary_topic,topics,locations,concepts,authorships,referenced_works,countries_distinct_count,keywords,counts_by_year
0,https://openalex.org/W4323655724,https://doi.org/10.1016/j.lindif.2023.102274,ChatGPT for good? On opportunities and challen...,NaN,2023,2023-03-09,"{'is_oa': True, 'oa_status': 'green', 'oa_url'...",article,en,3628,...,{'id': 'pmh:oai:mediatum.ub.tum.de:node/170990...,"{'id': 'https://openalex.org/T11636', 'display...","[{'id': 'https://openalex.org/T11636', 'displa...","[{'id': 'doi:10.1016/j.lindif.2023.102274', 'i...","[{'id': 'https://openalex.org/C47177190', 'wik...","[{'author_position': 'first', 'author': {'id':...","['https://openalex.org/W4312193283', 'https://...",1,[{'id': 'https://openalex.org/keywords/curricu...,"[{'year': 2025, 'cited_by_count': 1524}, {'yea..."
1,https://openalex.org/W3000065748,https://doi.org/10.1002/widm.1355,Educational data mining and learning analytics...,"{'Abstract': [0], 'This': [1, 93, 140], 'surve...",2020,2020-01-13,"{'is_oa': True, 'oa_status': 'green', 'oa_url'...",article,en,827,...,"{'id': 'pmh:oai:arXiv.org:2402.07956', 'is_oa'...","{'id': 'https://openalex.org/T11122', 'display...","[{'id': 'https://openalex.org/T11122', 'displa...","[{'id': 'doi:10.1002/widm.1355', 'is_oa': Fals...","[{'id': 'https://openalex.org/C2522767166', 'w...","[{'author_position': 'first', 'author': {'id':...","['https://openalex.org/W2121342934', 'https://...",1,[{'id': 'https://openalex.org/keywords/data-sc...,"[{'year': 2025, 'cited_by_count': 166}, {'year..."
2,https://openalex.org/W4304943299,https://doi.org/10.1007/s10639-022-11316-w,Ethical principles for artificial intelligence...,NaN,2022,2022-10-13,"{'is_oa': True, 'oa_status': 'hybrid', 'oa_url...",article,en,795,...,"{'id': 'doi:10.1007/s10639-022-11316-w', 'is_o...","{'id': 'https://openalex.org/T11122', 'display...","[{'id': 'https://openalex.org/T11122', 'displa...","[{'id': 'doi:10.1007/s10639-022-11316-w', 'is_...","[{'id': 'https://openalex.org/C65414064', 'wik...","[{'author_position': 'first', 'author': {'id':...","['https://openalex.org/W3199189488', 'https://...",3,[{'id': 'https://openalex.org/keywords/autonom...,"[{'year': 2025, 'cited_by_count': 411}, {'year..."
3,https://openalex.org/W4282940252,https://doi.org/10.3389/fpsyg.2022.813632,Lessons Learned and Future Directions of MetaT...,"{'Self-regulated': [0], 'learning': [1, 6, 35,...",2022,2022-06-14,"{'is_oa': True, 'oa_status': 'gold', 'oa_url':...",review,en,144,...,"{'id': 'doi:10.3389/fpsyg.2022.813632', 'is_oa...","{'id': 'https://openalex.org/T10636', 'display...","[{'id': 'https://openalex.org/T10636', 'displa...","[{'id': 'doi:10.3389/fpsyg.2022.813632', 'is_o...","[{'id': 'https://openalex.org/C118147538', 'wi...","[{'author_position': 'first', 'author': {'id':...","['https://openalex.org/W2626857346', 'https://...",5,[{'id': 'https://openalex.org/keywords/metacog...,"[{'year': 2025, 'cited_by_count': 60}, {'year'..."
4,https://openalex.org/W3155263273,https://doi.org/10.1007/s40593-021-00239-1,Ethics of AI in Education: Towards a Community...,"{'Abstract': [0], 'While': [1], 'Artificial': ...",2021,2021-04-09,"{'is_oa': True, 'oa_status': 'hybrid', 'oa_url...",article,en,770,...,"{'id': 'doi:10.1007/s40593-021-00239-1', 'is_o...","{'id': 'https://openalex.org/T11122', 'display...","[{'id': 'https://openalex.org/T11122', 'displa...","[{'id': 'doi:10.1007/s40593-021-00239-1', 'is_...","[{'id': 'https://openalex.org/C2776007630', 'w...","[{'author_position': 'first', 'author': {'id':...","['https://openalex.org/W2098788030', 'https://...",6,[{'id': 'https://openalex.org/keywords/account...,"[{'year': 2025, 'cited_by_count': 326}, {'year..."


In [13]:
#pefrom this operation for every column that contains list/dict-like strings
#find all columns with list/dict-like strings and apply ast.literal_eval
# abstract_inverted_index, open_acess, primary_location, best_oa_location, primary_topic, topics, locations, concepts, authorships, referenced_works, keywords, counts_by_year
# this script will convert those string representations back to their original list/dict formats

df["authorships"] = df["authorships"].apply(lambda x: ast.literal_eval(x) if pd.notnull(x) else None)
df["best_oa_location"] = df["best_oa_location"].apply(lambda x: ast.literal_eval(x) if pd.notnull(x) else None)
df["primary_location"] = df["primary_location"].apply(lambda x: ast.literal_eval(x) if pd.notnull(x) else None)
df["open_access"] = df["open_access"].apply(lambda x: ast.literal_eval(x) if pd.notnull(x) else None)
df["primary_topic"] = df["primary_topic"].apply(lambda x: ast.literal_eval(x) if pd.notnull(x) else None)
df["topics"] = df["topics"].apply(lambda x: ast.literal_eval(x) if pd.notnull(x) else None)
df["locations"] = df["locations"].apply(lambda x: ast.literal_eval(x) if pd.notnull(x) else None)
df["concepts"] = df["concepts"].apply(lambda x: ast.literal_eval(x) if pd.notnull(x) else None)
df["referenced_works"] = df["referenced_works"].apply(lambda x: ast.literal_eval(x) if pd.notnull(x) else None)
df["keywords"] = df["keywords"].apply(lambda x: ast.literal_eval(x) if pd.notnull(x) else None)
df["counts_by_year"] = df["counts_by_year"].apply(lambda x: ast.literal_eval(x) if pd.notnull(x) else None)
df["abstract_inverted_index"] = df["abstract_inverted_index"].apply(lambda x: ast.literal_eval(x) if pd.notnull(x) else None)

df.to_csv("open_alex_raw_1.csv", index=False)






#### Has EU affiliation

In [14]:
df = pd.read_csv("open_alex_raw_1.csv")


# from each authorship field, check if they have EU affiliation. Use the fallback method of raw affiliation strings if no institution data is present.
# add a new column to the dataframe indicating if any author has EU affiliation (True/False)
def has_eu_affiliation(authorships):
    # handle missing / NaN
    if pd.isna(authorships) or authorships is None:
        return False

    # if string, try to parse it (some rows store stringified lists)
    if isinstance(authorships, str):
        try:
            authorships = ast.literal_eval(authorships)
        except Exception:
            return False

    # if still not a list, nothing to check
    if not isinstance(authorships, list):
        return False

    for a in authorships:
        # skip non-dict items
        if not isinstance(a, dict):
            continue

        insts = a.get("institutions", []) or []
        for inst in insts:
            if not isinstance(inst, dict):
                continue
            country_code = (inst.get("country_code") or "").upper()
            if country_code in EU_CODES:
                return True

        # Fallback to raw affiliation strings
        raw_affils = a.get("raw_affiliation_strings", []) or []
        for raw_affil in raw_affils:
            if not isinstance(raw_affil, str):
                continue
            raw_upper = raw_affil.upper()
            for code in EU_CODES:
                if code in raw_upper:
                    return True

    return False

df["has_eu_affiliation"] = df["authorships"].apply(has_eu_affiliation)

# check how many rows have EU affiliation
eu_count = df["has_eu_affiliation"].sum()
print(f"Number of works with at least one EU-affiliated author: {eu_count} out of {len(df)}")



Number of works with at least one EU-affiliated author: 9689 out of 9689


In [15]:
df.head()

,id,doi,title,abstract_inverted_index,publication_year,publication_date,open_access,type,language,cited_by_count,...,primary_topic,topics,locations,concepts,authorships,referenced_works,countries_distinct_count,keywords,counts_by_year,has_eu_affiliation
0,https://openalex.org/W4323655724,https://doi.org/10.1016/j.lindif.2023.102274,ChatGPT for good? On opportunities and challen...,NaN,2023,2023-03-09,"{'is_oa': True, 'oa_status': 'green', 'oa_url'...",article,en,3628,...,"{'id': 'https://openalex.org/T11636', 'display...","[{'id': 'https://openalex.org/T11636', 'displa...","[{'id': 'doi:10.1016/j.lindif.2023.102274', 'i...","[{'id': 'https://openalex.org/C47177190', 'wik...","[{'author_position': 'first', 'author': {'id':...","['https://openalex.org/W4312193283', 'https://...",1,[{'id': 'https://openalex.org/keywords/curricu...,"[{'year': 2025, 'cited_by_count': 1524}, {'yea...",True
1,https://openalex.org/W3000065748,https://doi.org/10.1002/widm.1355,Educational data mining and learning analytics...,"{'Abstract': [0], 'This': [1, 93, 140], 'surve...",2020,2020-01-13,"{'is_oa': True, 'oa_status': 'green', 'oa_url'...",article,en,827,...,"{'id': 'https://openalex.org/T11122', 'display...","[{'id': 'https://openalex.org/T11122', 'displa...","[{'id': 'doi:10.1002/widm.1355', 'is_oa': Fals...","[{'id': 'https://openalex.org/C2522767166', 'w...","[{'author_position': 'first', 'author': {'id':...","['https://openalex.org/W2121342934', 'https://...",1,[{'id': 'https://openalex.org/keywords/data-sc...,"[{'year': 2025, 'cited_by_count': 166}, {'year...",True
2,https://openalex.org/W4304943299,https://doi.org/10.1007/s10639-022-11316-w,Ethical principles for artificial intelligence...,NaN,2022,2022-10-13,"{'is_oa': True, 'oa_status': 'hybrid', 'oa_url...",article,en,795,...,"{'id': 'https://openalex.org/T11122', 'display...","[{'id': 'https://openalex.org/T11122', 'displa...","[{'id': 'doi:10.1007/s10639-022-11316-w', 'is_...","[{'id': 'https://openalex.org/C65414064', 'wik...","[{'author_position': 'first', 'author': {'id':...","['https://openalex.org/W3199189488', 'https://...",3,[{'id': 'https://openalex.org/keywords/autonom...,"[{'year': 2025, 'cited_by_count': 411}, {'year...",True
3,https://openalex.org/W4282940252,https://doi.org/10.3389/fpsyg.2022.813632,Lessons Learned and Future Directions of MetaT...,"{'Self-regulated': [0], 'learning': [1, 6, 35,...",2022,2022-06-14,"{'is_oa': True, 'oa_status': 'gold', 'oa_url':...",review,en,144,...,"{'id': 'https://openalex.org/T10636', 'display...","[{'id': 'https://openalex.org/T10636', 'displa...","[{'id': 'doi:10.3389/fpsyg.2022.813632', 'is_o...","[{'id': 'https://openalex.org/C118147538', 'wi...","[{'author_position': 'first', 'author': {'id':...","['https://openalex.org/W2626857346', 'https://...",5,[{'id': 'https://openalex.org/keywords/metacog...,"[{'year': 2025, 'cited_by_count': 60}, {'year'...",True
4,https://openalex.org/W3155263273,https://doi.org/10.1007/s40593-021-00239-1,Ethics of AI in Education: Towards a Community...,"{'Abstract': [0], 'While': [1], 'Artificial': ...",2021,2021-04-09,"{'is_oa': True, 'oa_status': 'hybrid', 'oa_url...",article,en,770,...,"{'id': 'https://openalex.org/T11122', 'display...","[{'id': 'https://openalex.org/T11122', 'displa...","[{'id': 'doi:10.1007/s40593-021-00239-1', 'is_...","[{'id': 'https://openalex.org/C2776007630', 'w...","[{'author_position': 'first', 'author': {'id':...","['https://openalex.org/W2098788030', 'https://...",6,[{'id': 'https://openalex.org/keywords/account...,"[{'year': 2025, 'cited_by_count': 326}, {'year...",True


#### has 2 or more distinct institutions

In [16]:
#check whether there are 2 or more distinct institutions in the authorships. 
#check ["institutions"]["id"] for each author in authorships. 
# add a new column to the dataframe indicating if there are 2 or more distinct institutions (True/False)
def has_multiple_institutions(authorships):
    if pd.isna(authorships) or authorships is None:
        return False

    if isinstance(authorships, str):
        try:
            authorships = ast.literal_eval(authorships)
        except Exception:
            return False

    if not isinstance(authorships, list):
        return False

    institution_ids = set()
    for a in authorships:
        if not isinstance(a, dict):
            continue

        insts = a.get("institutions", []) or []
        for inst in insts:
            if not isinstance(inst, dict):
                continue
            inst_id = inst.get("id")
            if inst_id:
                institution_ids.add(inst_id)

    return len(institution_ids) >= 2
df["has_multiple_institutions"] = df["authorships"].apply(has_multiple_institutions)
#list the name of institutions if there are 2 or more distinct institutions in a new column
def list_distinct_institutions(authorships):
    if pd.isna(authorships) or authorships is None:
        return []

    if isinstance(authorships, str):
        try:
            authorships = ast.literal_eval(authorships)
        except Exception:
            return []

    if not isinstance(authorships, list):
        return []

    institution_names = set()
    for a in authorships:
        if not isinstance(a, dict):
            continue

        insts = a.get("institutions", []) or []
        for inst in insts:
            if not isinstance(inst, dict):
                continue
            inst_name = inst.get("display_name")
            if inst_name:
                institution_names.add(inst_name)

    return list(institution_names)

df["distinct_institutions"] = df["authorships"].apply(list_distinct_institutions)

In [17]:
df.head()

,id,doi,title,abstract_inverted_index,publication_year,publication_date,open_access,type,language,cited_by_count,...,locations,concepts,authorships,referenced_works,countries_distinct_count,keywords,counts_by_year,has_eu_affiliation,has_multiple_institutions,distinct_institutions
0,https://openalex.org/W4323655724,https://doi.org/10.1016/j.lindif.2023.102274,ChatGPT for good? On opportunities and challen...,NaN,2023,2023-03-09,"{'is_oa': True, 'oa_status': 'green', 'oa_url'...",article,en,3628,...,"[{'id': 'doi:10.1016/j.lindif.2023.102274', 'i...","[{'id': 'https://openalex.org/C47177190', 'wik...","[{'author_position': 'first', 'author': {'id':...","['https://openalex.org/W4312193283', 'https://...",1,[{'id': 'https://openalex.org/keywords/curricu...,"[{'year': 2025, 'cited_by_count': 1524}, {'yea...",True,True,"[Ludwig-Maximilians-Universität München, Techn..."
1,https://openalex.org/W3000065748,https://doi.org/10.1002/widm.1355,Educational data mining and learning analytics...,"{'Abstract': [0], 'This': [1, 93, 140], 'surve...",2020,2020-01-13,"{'is_oa': True, 'oa_status': 'green', 'oa_url'...",article,en,827,...,"[{'id': 'doi:10.1002/widm.1355', 'is_oa': Fals...","[{'id': 'https://openalex.org/C2522767166', 'w...","[{'author_position': 'first', 'author': {'id':...","['https://openalex.org/W2121342934', 'https://...",1,[{'id': 'https://openalex.org/keywords/data-sc...,"[{'year': 2025, 'cited_by_count': 166}, {'year...",True,True,"[University of Córdoba, Universidad Laboral de..."
2,https://openalex.org/W4304943299,https://doi.org/10.1007/s10639-022-11316-w,Ethical principles for artificial intelligence...,NaN,2022,2022-10-13,"{'is_oa': True, 'oa_status': 'hybrid', 'oa_url...",article,en,795,...,"[{'id': 'doi:10.1007/s10639-022-11316-w', 'is_...","[{'id': 'https://openalex.org/C65414064', 'wik...","[{'author_position': 'first', 'author': {'id':...","['https://openalex.org/W3199189488', 'https://...",3,[{'id': 'https://openalex.org/keywords/autonom...,"[{'year': 2025, 'cited_by_count': 411}, {'year...",True,True,"[Victoria University of Wellington, University..."
3,https://openalex.org/W4282940252,https://doi.org/10.3389/fpsyg.2022.813632,Lessons Learned and Future Directions of MetaT...,"{'Self-regulated': [0], 'learning': [1, 6, 35,...",2022,2022-06-14,"{'is_oa': True, 'oa_status': 'gold', 'oa_url':...",review,en,144,...,"[{'id': 'doi:10.3389/fpsyg.2022.813632', 'is_o...","[{'id': 'https://openalex.org/C118147538', 'wi...","[{'author_position': 'first', 'author': {'id':...","['https://openalex.org/W2626857346', 'https://...",5,[{'id': 'https://openalex.org/keywords/metacog...,"[{'year': 2025, 'cited_by_count': 60}, {'year'...",True,True,"[Sorbonne Université, Universität Greifswald, ..."
4,https://openalex.org/W3155263273,https://doi.org/10.1007/s40593-021-00239-1,Ethics of AI in Education: Towards a Community...,"{'Abstract': [0], 'While': [1], 'Artificial': ...",2021,2021-04-09,"{'is_oa': True, 'oa_status': 'hybrid', 'oa_url...",article,en,770,...,"[{'id': 'doi:10.1007/s40593-021-00239-1', 'is_...","[{'id': 'https://openalex.org/C2776007630', 'w...","[{'author_position': 'first', 'author': {'id':...","['https://openalex.org/W2098788030', 'https://...",6,[{'id': 'https://openalex.org/keywords/account...,"[{'year': 2025, 'cited_by_count': 326}, {'year...",True,True,[Universidad Nacional de Educación a Distancia...


#### has 2 or more distinct country codes on a paper.

In [18]:
# check whether there are 2 or more distinct countries in the authorships. 
# check ["institutions"]["country_code"] and add a new column to the dataframe indicating if there are 2 or more distinct countries (True/False)
def has_multiple_countries(authorships):
    if pd.isna(authorships) or authorships is None:
        return False

    if isinstance(authorships, str):
        try:
            authorships = ast.literal_eval(authorships)
        except Exception:
            return False

    if not isinstance(authorships, list):
        return False

    country_codes = set()
    for a in authorships:
        if not isinstance(a, dict):
            continue

        insts = a.get("institutions", []) or []
        for inst in insts:
            if not isinstance(inst, dict):
                continue
            country_code = inst.get("country_code")
            if country_code:
                country_codes.add(country_code)

    return len(country_codes) >= 2

df["has_multiple_countries"] = df["authorships"].apply(has_multiple_countries)

#list the name of the countries if there are 2 or more distinct countries in a new column
def list_distinct_countries(authorships):
    if pd.isna(authorships) or authorships is None:
        return []

    if isinstance(authorships, str):
        try:
            authorships = ast.literal_eval(authorships)
        except Exception:
            return []

    if not isinstance(authorships, list):
        return []

    country_codes = set()
    for a in authorships:
        if not isinstance(a, dict):
            continue

        insts = a.get("institutions", []) or []
        for inst in insts:
            if not isinstance(inst, dict):
                continue
            country_code = inst.get("country_code")
            if country_code:
                country_codes.add(country_code)

    return list(country_codes)
df["distinct_countries"] = df["authorships"].apply(list_distinct_countries)


In [19]:
df.head()

,id,doi,title,abstract_inverted_index,publication_year,publication_date,open_access,type,language,cited_by_count,...,authorships,referenced_works,countries_distinct_count,keywords,counts_by_year,has_eu_affiliation,has_multiple_institutions,distinct_institutions,has_multiple_countries,distinct_countries
0,https://openalex.org/W4323655724,https://doi.org/10.1016/j.lindif.2023.102274,ChatGPT for good? On opportunities and challen...,NaN,2023,2023-03-09,"{'is_oa': True, 'oa_status': 'green', 'oa_url'...",article,en,3628,...,"[{'author_position': 'first', 'author': {'id':...","['https://openalex.org/W4312193283', 'https://...",1,[{'id': 'https://openalex.org/keywords/curricu...,"[{'year': 2025, 'cited_by_count': 1524}, {'yea...",True,True,"[Ludwig-Maximilians-Universität München, Techn...",False,[DE]
1,https://openalex.org/W3000065748,https://doi.org/10.1002/widm.1355,Educational data mining and learning analytics...,"{'Abstract': [0], 'This': [1, 93, 140], 'surve...",2020,2020-01-13,"{'is_oa': True, 'oa_status': 'green', 'oa_url'...",article,en,827,...,"[{'author_position': 'first', 'author': {'id':...","['https://openalex.org/W2121342934', 'https://...",1,[{'id': 'https://openalex.org/keywords/data-sc...,"[{'year': 2025, 'cited_by_count': 166}, {'year...",True,True,"[University of Córdoba, Universidad Laboral de...",False,[ES]
2,https://openalex.org/W4304943299,https://doi.org/10.1007/s10639-022-11316-w,Ethical principles for artificial intelligence...,NaN,2022,2022-10-13,"{'is_oa': True, 'oa_status': 'hybrid', 'oa_url...",article,en,795,...,"[{'author_position': 'first', 'author': {'id':...","['https://openalex.org/W3199189488', 'https://...",3,[{'id': 'https://openalex.org/keywords/autonom...,"[{'year': 2025, 'cited_by_count': 411}, {'year...",True,True,"[Victoria University of Wellington, University...",True,"[FI, VN, NZ]"
3,https://openalex.org/W4282940252,https://doi.org/10.3389/fpsyg.2022.813632,Lessons Learned and Future Directions of MetaT...,"{'Self-regulated': [0], 'learning': [1, 6, 35,...",2022,2022-06-14,"{'is_oa': True, 'oa_status': 'gold', 'oa_url':...",review,en,144,...,"[{'author_position': 'first', 'author': {'id':...","['https://openalex.org/W2626857346', 'https://...",5,[{'id': 'https://openalex.org/keywords/metacog...,"[{'year': 2025, 'cited_by_count': 60}, {'year'...",True,True,"[Sorbonne Université, Universität Greifswald, ...",True,"[CA, FR, DE, ES, US]"
4,https://openalex.org/W3155263273,https://doi.org/10.1007/s40593-021-00239-1,Ethics of AI in Education: Towards a Community...,"{'Abstract': [0], 'While': [1], 'Artificial': ...",2021,2021-04-09,"{'is_oa': True, 'oa_status': 'hybrid', 'oa_url...",article,en,770,...,"[{'author_position': 'first', 'author': {'id':...","['https://openalex.org/W2098788030', 'https://...",6,[{'id': 'https://openalex.org/keywords/account...,"[{'year': 2025, 'cited_by_count': 326}, {'year...",True,True,[Universidad Nacional de Educación a Distancia...,True,"[BR, ES, GB, US, PH, AU]"


#### Working on topics

In [20]:
topics_list = df.loc[0, "topics"]

# If topics_list is a string (stringified list), parse it. Handle NaN/None safely.
if pd.isna(topics_list) or topics_list is None:
    topics_list = []
elif isinstance(topics_list, str):
    try:
        topics_list = ast.literal_eval(topics_list)
    except Exception:
        # fallback to empty list if parsing fails
        topics_list = []

# Print display_name and score (guard for non-dict items)
for t in topics_list:
    if isinstance(t, dict):
        name = t.get("display_name")
        score = t.get("score")
    else:
        name = None
        score = None
    print(name, score)


    # I want to get the display_names and scores of each element in topics.
# I want to create a new column "topics_parsed" in the dataframe that contains a list of dictionaries with display_name and score for each topic.
# Each dictionary should have the keys 'display_name' and 'score'. 
def parse_topics(topics):
    if pd.isna(topics) or topics is None:
        return []

    if isinstance(topics, str):
        try:
            topics = ast.literal_eval(topics)
        except Exception:
            return []

    if not isinstance(topics, list):
        return []

    parsed_topics = []
    for t in topics:
        if isinstance(t, dict):
            name = t.get("display_name")
            score = t.get("score")
            parsed_topics.append({"display_name": name, "score": score})
    return parsed_topics
df["topics_parsed"] = df["topics"].apply(parse_topics)

Artificial Intelligence in Healthcare and Education 0.9825000166893005
Topic Modeling 0.9814000129699707
Online Learning and Analytics 0.9753000140190125


In [21]:
df.head()
#print entries from topics_parsed column
for entry in df["topics_parsed"].iloc[0]:
    print(entry)

{'display_name': 'Artificial Intelligence in Healthcare and Education', 'score': 0.9825000166893005}
{'display_name': 'Topic Modeling', 'score': 0.9814000129699707}
{'display_name': 'Online Learning and Analytics', 'score': 0.9753000140190125}


#### concepts_list (cleaned semicolon list; drop generic “other”)

In [22]:
#print an item from the topics columns and one from the concepts columns
# Print number of elements in topics and concepts for the first row
first_row = df.iloc[0]
topics = first_row["topics"]
concepts = first_row["concepts"]


In [23]:
#just like the topics, I want to print the concepts with display_name, level, and score.
def parse_concepts(concepts):
    if pd.isna(concepts) or concepts is None:
        return []

    if isinstance(concepts, str):
        try:
            concepts = ast.literal_eval(concepts)
        except Exception:
            return []
    if not isinstance(concepts, list):
        return []
    parsed_concepts = []
    for c in concepts:
        if isinstance(c, dict):
            name = c.get("display_name")
            level = c.get("level")
            score = c.get("score")
            parsed_concepts.append({"display_name": name, "level": level, "score": score})
    return parsed_concepts
df["concepts_parsed"] = df["concepts"].apply(parse_concepts)

In [24]:
df.head()
for entry in df["concepts_parsed"].iloc[0]:
    print(entry)

{'display_name': 'Curriculum', 'level': 2, 'score': 0.5452895164489746}
{'display_name': 'Computer science', 'level': 0, 'score': 0.5172855854034424}
{'display_name': 'Field (mathematics)', 'level': 2, 'score': 0.46225565671920776}
{'display_name': 'Engineering ethics', 'level': 1, 'score': 0.4140339493751526}
{'display_name': 'Knowledge management', 'level': 1, 'score': 0.336037814617157}
{'display_name': 'Management science', 'level': 1, 'score': 0.3205817937850952}
{'display_name': 'Psychology', 'level': 0, 'score': 0.27662089467048645}
{'display_name': 'Pedagogy', 'level': 1, 'score': 0.22286361455917358}
{'display_name': 'Engineering', 'level': 0, 'score': 0.12583649158477783}
{'display_name': 'Pure mathematics', 'level': 1, 'score': 0.0}
{'display_name': 'Mathematics', 'level': 0, 'score': 0.0}


#### issn, issl

In [25]:
def extract_all_issns(primary_location):
    # handle missing / NaN
    if pd.isna(primary_location) or primary_location is None:
        return []

    # if string (stringified dict), try to parse
    if isinstance(primary_location, str):
        try:
            primary_location = ast.literal_eval(primary_location)
        except Exception:
            return []

    # must be a dict now
    if not isinstance(primary_location, dict):
        return []

    source = primary_location.get("source") or {}
    if not isinstance(source, dict):
        return []

    # issn_l may be a string; issn may be list or string or None
    issn_l = source.get("issn_l")
    issn_list = source.get("issn")

    all_issns = set()

    if issn_l:
        all_issns.add(str(issn_l).replace("-", "").strip())

    # normalize issn_list to iterable
    if issn_list:
        if isinstance(issn_list, str):
            # sometimes a single ISSN is stored as a string
            issn_iter = [issn_list]
        elif isinstance(issn_list, (list, tuple, set)):
            issn_iter = issn_list
        else:
            # unexpected type
            issn_iter = []
    else:
        issn_iter = []

    for issn in issn_iter:
        if issn:
            all_issns.add(str(issn).replace("-", "").strip())

    return list(all_issns)

# apply it
df["all_issns"] = df["primary_location"].apply(extract_all_issns)

In [26]:

for entry in df["all_issns"].iloc[0]:
	print(entry)

10416080
18733425


In [27]:
df.head()

,id,doi,title,abstract_inverted_index,publication_year,publication_date,open_access,type,language,cited_by_count,...,keywords,counts_by_year,has_eu_affiliation,has_multiple_institutions,distinct_institutions,has_multiple_countries,distinct_countries,topics_parsed,concepts_parsed,all_issns
0,https://openalex.org/W4323655724,https://doi.org/10.1016/j.lindif.2023.102274,ChatGPT for good? On opportunities and challen...,NaN,2023,2023-03-09,"{'is_oa': True, 'oa_status': 'green', 'oa_url'...",article,en,3628,...,[{'id': 'https://openalex.org/keywords/curricu...,"[{'year': 2025, 'cited_by_count': 1524}, {'yea...",True,True,"[Ludwig-Maximilians-Universität München, Techn...",False,[DE],[{'display_name': 'Artificial Intelligence in ...,"[{'display_name': 'Curriculum', 'level': 2, 's...","[10416080, 18733425]"
1,https://openalex.org/W3000065748,https://doi.org/10.1002/widm.1355,Educational data mining and learning analytics...,"{'Abstract': [0], 'This': [1, 93, 140], 'surve...",2020,2020-01-13,"{'is_oa': True, 'oa_status': 'green', 'oa_url'...",article,en,827,...,[{'id': 'https://openalex.org/keywords/data-sc...,"[{'year': 2025, 'cited_by_count': 166}, {'year...",True,True,"[University of Córdoba, Universidad Laboral de...",False,[ES],[{'display_name': 'Online Learning and Analyti...,"[{'display_name': 'Data science', 'level': 1, ...","[19424795, 19424787]"
2,https://openalex.org/W4304943299,https://doi.org/10.1007/s10639-022-11316-w,Ethical principles for artificial intelligence...,NaN,2022,2022-10-13,"{'is_oa': True, 'oa_status': 'hybrid', 'oa_url...",article,en,795,...,[{'id': 'https://openalex.org/keywords/autonom...,"[{'year': 2025, 'cited_by_count': 411}, {'year...",True,True,"[Victoria University of Wellington, University...",True,"[FI, VN, NZ]",[{'display_name': 'Online Learning and Analyti...,"[{'display_name': 'Autonomy', 'level': 2, 'sco...","[13602357, 15737608]"
3,https://openalex.org/W4282940252,https://doi.org/10.3389/fpsyg.2022.813632,Lessons Learned and Future Directions of MetaT...,"{'Self-regulated': [0], 'learning': [1, 6, 35,...",2022,2022-06-14,"{'is_oa': True, 'oa_status': 'gold', 'oa_url':...",review,en,144,...,[{'id': 'https://openalex.org/keywords/metacog...,"[{'year': 2025, 'cited_by_count': 60}, {'year'...",True,True,"[Sorbonne Université, Universität Greifswald, ...",True,"[CA, FR, DE, ES, US]",[{'display_name': 'Innovative Teaching and Lea...,"[{'display_name': 'Metacognition', 'level': 3,...",[16641078]
4,https://openalex.org/W3155263273,https://doi.org/10.1007/s40593-021-00239-1,Ethics of AI in Education: Towards a Community...,"{'Abstract': [0], 'While': [1], 'Artificial': ...",2021,2021-04-09,"{'is_oa': True, 'oa_status': 'hybrid', 'oa_url...",article,en,770,...,[{'id': 'https://openalex.org/keywords/account...,"[{'year': 2025, 'cited_by_count': 326}, {'year...",True,True,[Universidad Nacional de Educación a Distancia...,True,"[BR, ES, GB, US, PH, AU]",[{'display_name': 'Online Learning and Analyti...,"[{'display_name': 'Accountability', 'level': 2...","[15604292, 15604306]"


In [28]:
#save the csv
df.to_csv("open_alex_raw_flagged.csv", index=False)